In [1]:
!pip install keybert sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer, util as st_util
from IPython.display import display, HTML

In [3]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH    = '/content/drive/MyDrive/Licenta/medical_dataset_final.csv'

df = pd.read_csv(CSV_PATH, sep=';', on_bad_lines='skip')
df = df[['filename', 'rezumat', 'keywords']].copy()
df['rezumat'] = df['rezumat'].fillna('')

print(f"Date încărcate: {len(df)} articole")

Mounted at /content/drive
Date încărcate: 1114 articole


In [4]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f"Se încarcă modelul {MODEL_NAME}...")
kw_model  = KeyBERT(model=MODEL_NAME)
sem_model = SentenceTransformer(MODEL_NAME)  # refolosit la evaluare
print("Model încărcat.")

Se încarcă modelul paraphrase-multilingual-MiniLM-L12-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model încărcat.


In [14]:
TOP_N = 5

def extract_keybert(text, n=TOP_N):
    if not text or len(text.strip()) < 20:
        return []
    return kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 1),
        stop_words=None,
        use_mmr=True,
        diversity=0.3,
        top_n=n
    )

print("Extrag keywords cu KeyBERT (poate dura 2-3 minute)...")
results = []
for i, row in df.iterrows():
    kw = extract_keybert(row['rezumat'])
    results.append({
        'filename'           : row['filename'],
        'keywords_extracted' : '; '.join([k for k, s in kw]),
        'keywords_original'  : row['keywords'],
        'keybert_scores'     : str([(k, round(s, 4)) for k, s in kw])
    })

results_df = pd.DataFrame(results)
print(f"Keywords extrase pentru {len(results_df)} articole")

Extrag keywords cu KeyBERT (poate dura 2-3 minute)...
Keywords extrase pentru 1114 articole


In [15]:
SIM_THRESHOLD = 0.6

def parse_keywords(text):
    if pd.isna(text) or str(text).strip() == '':
        return []
    return [w.strip().lower() for w in re.split(r'[;,]', str(text)) if w.strip()]

def compute_metrics(extracted_str, original_str):
    extracted = parse_keywords(extracted_str)
    original  = parse_keywords(original_str)

    if not extracted or not original:
        return 0.0, 0.0, 0.0

    all_terms  = extracted + original
    embeddings = sem_model.encode(all_terms, convert_to_tensor=True)
    emb_ext    = embeddings[:len(extracted)]
    emb_orig   = embeddings[len(extracted):]

    matched_orig = set()  # fiecare keyword original se poate potrivi o singură dată
    tp = 0
    for i, e in enumerate(extracted):
        for j, o in enumerate(original):
            if j in matched_orig:
                continue
            string_match   = e in o or o in e
            cos_score      = st_util.cos_sim(emb_ext[i], emb_orig[j]).item()
            semantic_match = cos_score >= SIM_THRESHOLD
            if string_match or semantic_match:
                tp += 1
                matched_orig.add(j)
                break

    precision = tp / len(extracted)
    recall    = tp / len(original)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print("Calculez metrici semantice (poate dura 1-2 minute)...")
results_df[['precision', 'recall', 'f1']] = results_df.apply(
    lambda r: pd.Series(compute_metrics(r['keywords_extracted'], r['keywords_original'])),
    axis=1
)

avg_p  = results_df['precision'].mean()
avg_r  = results_df['recall'].mean()
avg_f1 = results_df['f1'].mean()

print(f"\n{'='*45}")
print(f"  Precision medie : {avg_p:.4f}  ({avg_p:.2%})")
print(f"  Recall mediu    : {avg_r:.4f}  ({avg_r:.2%})")
print(f"  F1 mediu        : {avg_f1:.4f}  ({avg_f1:.2%})")
print(f"{'='*45}")
print(f"  Documente cu F1 > 0 : {(results_df['f1'] > 0).sum()} / {len(results_df)}")

Calculez metrici semantice (poate dura 1-2 minute)...

  Precision medie : 0.3698  (36.98%)
  Recall mediu    : 0.5296  (52.96%)
  F1 mediu        : 0.4247  (42.47%)
  Documente cu F1 > 0 : 1063 / 1114


In [16]:
N_PREVIEW = 20

def make_html_table(df_preview):
    rows = ''
    for _, r in df_preview.iterrows():
        color = '#c8f7c5' if r['f1'] >= 0.5 else '#fde8d8' if r['f1'] > 0 else '#f5f5f5'
        rows += f"""
        <tr>
            <td style='font-size:12px;color:#555;padding:6px 8px;'>{r['filename']}</td>
            <td style='color:#0a7c3e;font-weight:500;padding:6px 8px;'>{r['keywords_extracted']}</td>
            <td style='color:#555;padding:6px 8px;'>{r['keywords_original']}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['precision']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;'>{r['recall']:.2f}</td>
            <td style='text-align:center;padding:6px 8px;background:{color};font-weight:bold;'>{r['f1']:.2f}</td>
        </tr>"""

    html = f"""
    <h3 style='font-family:sans-serif;'>🔍 KeyBERT — Primele {N_PREVIEW} articole</h3>
    <table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>
        <thead>
            <tr style='background:#0a7c3e;color:white;'>
                <th style='padding:8px;text-align:left;'>Fișier</th>
                <th style='padding:8px;text-align:left;'>Keywords extrase (KeyBERT)</th>
                <th style='padding:8px;text-align:left;'>Keywords originale</th>
                <th style='padding:8px;text-align:center;'>Precision</th>
                <th style='padding:8px;text-align:center;'>Recall</th>
                <th style='padding:8px;text-align:center;'>F1</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style='font-family:sans-serif;font-size:12px;color:#888;margin-top:8px;'>
        🟢 F1 ≥ 0.5 &nbsp;|&nbsp; 🟠 F1 > 0 &nbsp;|&nbsp; ⚪ F1 = 0
    </p>
    """
    return html

display(HTML(make_html_table(results_df.head(N_PREVIEW))))

Fișier,Keywords extrase (KeyBERT),Keywords originale,Precision,Recall,F1
RMJ_2017_3_Art-08.pdf,statistic; pacienților; populație; colonoscopii; screening,tumori; maligne; studiu; retrospectiv; endoscopic,0.00,0.00,0.00
RMR_Nr-3_2011_Art-6.pdf,testul; testele; teste; diagnosticarea; pacienți,sindrom impingement; umăr; teste clinice; regiune deltoidiana,0.40,0.50,0.44
RJP_2017_2_RO_Art-05.pdf,tumoră; pediatrică; neuroectodermică; diagnosticare; meduloblastomul,meduloblastom; copil,0.40,1.00,0.57
RJP_2012_1_Art-09.pdf,hipoglicemia; hipoglicemiilor; metabolice; hormonale; copilărie,hipoglicemie persistentă; deﬁcit gh; tratament substitutiv,0.20,0.33,0.25
RJP_2015_4_RO_Art-07.pdf,fibrozei; tratamentul; screeningul; etică; transplantul,fibroză chistică; etică; screening; terapie genică; transplant; fertilitate,1.00,0.83,0.91
RJS_2014_3_Art-09.pdf,hemangioamele; hemangioamelor; hemangioame; tumori; benigne,hemangiom; cavitate orală; perioral; propranolol; corticosteroizi; laser; intralezional; scleroterapie,0.20,0.12,0.15
RJP_2012_2_Art-13.pdf,arterială; hidronefroză; hipertensiune; rare; tumoră,hidronefroză gigantă; hipertensiune; copil,0.40,0.67,0.50
RJP_2012_2_Art-07.pdf,wheezing; diagnosticului; copii; vârstă; respiratorii,wheezing recurent; copil; factori de risc; atopie,0.40,0.50,0.44
RJP_2011_1_Art-01.pdf,alergiei; alergologice; diagnostic; patogeniei; alimentară,alergie alimentară; diagnostic; copil,0.40,0.67,0.50
RMJ_2019_1_Art-11.pdf,diabetului; diabetul; insulinorezistenței; dz2; dislipidemia,diabet zaharat tip 2; tiroidită; autoimun; corelație,0.20,0.25,0.22
